## Contagem de erros ortográficos e gramaticais

Para o critério de ortografia, usamos a biblioteca `language_tool_python`,
que roda o LanguageTool — uma ferramenta de checagem gramatical que
suporta português (pt-BR). Ela recebe o texto de uma notícia e devolve
uma lista de possíveis erros: ortografia, concordância, pontuação e
outras inconsistências gramaticais.

In [ ]:
!pip install language_tool_python

In [ ]:
import language_tool_python

In [ ]:
# Necessário ter JAVA instalado
tool = language_tool_python.LanguageTool("pt-BR")

### Como funciona

A função `contar_erros()` faz três coisas:

1. Envia o texto da notícia para o LanguageTool (`tool.check(texto)`),
   que devolve uma lista de erros encontrados.
2. Conta quantos itens vieram nessa lista (`qtd_erros`).
3. Conta quantas palavras o texto tem (`qtd_palavras`), para permitir
   comparar notícias de tamanhos diferentes de forma justa.

### Por que normalizar por 100 palavras

Comparar apenas o número bruto de erros é enganoso, porque um texto
maior naturalmente tem mais chance de conter erros — não porque está
pior escrito, mas porque tem mais palavras onde um erro pode acontecer.

Por isso calculamos uma taxa, e não só a contagem crua:

    erros_por_100_palavras = (qtd_erros / qtd_palavras) * 100

**Exemplo:**

| Notícia | Erros | Palavras | Erros por 100 palavras |
|---|---|---|---|
| A | 3 | 50 | 6,0 |
| B | 3 | 500 | 0,6 |

As duas notícias têm o mesmo número de erros (3), mas a notícia A é,
proporcionalmente, **10 vezes pior escrita** que a B. Sem a
normalização, o dataset trataria as duas como igualmente ruins nesse
critério, o que distorceria o resultado.

In [ ]:
def contar_erros(texto: str) -> dict:
    """
    Recebe o texto de uma noticia e devolve um dicionario com:
      - qtd_erros: numero bruto de erros encontrados
      - qtd_palavras: tamanho do texto em palavras (usado pra normalizar)
      - erros_por_100_palavras: taxa de erro normalizada pelo tamanho do texto
    """
    if not isinstance(texto, str) or not texto.strip():
        return {"qtd_erros": 0, "qtd_palavras": 0, "erros_por_100_palavras": 0.0}

    erros = tool.check(texto)
    qtd_erros = len(erros)
    qtd_palavras = len(texto.split())

    # normalizacao: erros a cada 100 palavras, evita que texto longo
    # pareca "pior" so por ter mais chance estatistica de erro
    erros_por_100_palavras = (qtd_erros / qtd_palavras * 100) if qtd_palavras > 0 else 0.0

    return {
        "qtd_erros": qtd_erros,
        "qtd_palavras": qtd_palavras,
        "erros_por_100_palavras": round(erros_por_100_palavras, 2),
    }

In [ ]:
noticias = [
    "Texto da notícia 1 aqui...",
    "Texto da noticia 2 aqui...",
    "Texto da noticia 3 aqui...",
]

for i, texto in enumerate(noticias, start=1):
    resultado = contar_erros(texto)
    print(f"\nNotícia {i}")
    print(f"  Erros encontrados: {resultado['qtd_erros']}")
    print(f"  Palavras no texto: {resultado['qtd_palavras']}")
    print(f"  Taxa (por 100 palavras): {resultado['erros_por_100_palavras']}")

tool.close()